In [ ]:
from getpass import getuser # Libaray to copy things
from pathlib import Path # Object oriented libary to deal with paths
from dask.utils import format_bytes
from distributed import Client, progress, wait # Libaray to orchestrate distributed resources
#from dask_jobqueue import SLURMCluster # Setting up distributed memories via slurm
import numpy as np # Pythons standard array library
import xarray as xr # Libary to work with labeled n-dimensional data
import dask # Distributed data libary
import dask.distributed
import multiprocessing
from dask.diagnostics import ProgressBar
import glob, os
from matplotlib.lines import Line2D

from matplotlib.patches import Patch
from scipy.ndimage import gaussian_filter1d
from subprocess import run, PIPE
import sys
import os
from sklearn.neighbors import KernelDensity
import gc
from concurrent.futures import ProcessPoolExecutor
import seaborn as sns
import warnings
warnings.filterwarnings(action='ignore')
dask.config.config.get('distributed').get('dashboard').update({'link':'{JUPYTERHUB_SERVICE_PREFIX}/proxy/{port}/status'})
import pandas as pd
from scipy.interpolate import CloughTocher2DInterpolator, LinearNDInterpolator, NearestNDInterpolator

In [ ]:
from matplotlib.ticker import (MultipleLocator, FormatStrFormatter, AutoMinorLocator)
from matplotlib import pyplot as plt # Standard Plotting library
from matplotlib import cm
from matplotlib.colors import ListedColormap, LinearSegmentedColormap
from matplotlib.ticker import (MultipleLocator, FormatStrFormatter, AutoMinorLocator)
from cartopy import crs as ccrs # Cartography library
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import cartopy.feature as cfeature
import matplotlib.ticker as mticker
import matplotlib.tri as tri

In [ ]:
#'''Provides information about the CPU and starts the dask client'''
ncpu = multiprocessing.cpu_count()
processes = False
nworker = 1
threads = ncpu // nworker
print(f"Number of CPUs: {ncpu}, number of threads: {threads}, number of workers: {nworker}, processes: {processes}")
client = Client(processes=processes,
 threads_per_worker=threads,
 n_workers=nworker,
 memory_limit='1000GB'
 )

In [ ]:
latitude = [30,60]


In [ ]:
ifs_on = xr.open_dataset('/home/jwille/xenon_data/NextGEMS/dry_spells/IFS_2.8-FESOM_5_deep_on_summer_2020_dry_spell_30-60N.nc')['__xarray_dataarray_variable__']
ifs_off = xr.open_dataset('/home/jwille/xenon_data/NextGEMS/dry_spells/IFS_2.8-FESOM_5_deep_off_summer_2020_dry_spell_30-60N.nc')['__xarray_dataarray_variable__']
ifs_cycle3 = xr.open_dataset('/home/jwille/xenon_data/NextGEMS/dry_spells/IFS_cycle3_summer_2020-2024_dry_spell_annual_30-60N.nc')['__xarray_dataarray_variable__']
ifs_cycle4 = xr.open_dataset('/home/jwille/xenon_data/NextGEMS/dry_spells/IFS_cycle4_summer_1990-2019_dry_spell_annual_30-60N.nc')['__xarray_dataarray_variable__']
icon_cycle3 = xr.open_dataset('/home/jwille/xenon_data/NextGEMS/dry_spells/ICON_zoom10_summer_2020-2024_dry_spell_annual_30-60N.nc')['__xarray_dataarray_variable__']
icon_cycle4 = xr.open_dataset('/home/jwille/xenon_data/NextGEMS/dry_spells/ICON_zoom9_summer_2020-2024_dry_spell_annual_30-60N.nc')['__xarray_dataarray_variable__']
imerg = xr.open_dataset('/home/jwille/xenon_data/NextGEMS/dry_spells/IMERG_summer_2014-2023_dry_spell_annual_30-60N.nc')['__xarray_dataarray_variable__']
#ifs_cycle4 = ifs_cycle4.isel(year=slice(0,30))
#ifs_cycle4 = ifs_cycle4.stack(sample=("year", "time"))
#icon_cycle4 = icon_cycle4.isel(year=slice(0,5))
#icon_cycle4 = icon_cycle4.stack(sample=("year", "time"))



In [ ]:
print(ifs_on)

In [ ]:
print(icon_cycle4)

In [ ]:
print(ifs_cycle4.lat.values.max())

In [ ]:
print(icon_cycle3)

In [ ]:
land_sea_ifs_high = xr.open_dataset('/home/jwille/xenon_data/NextGEMS/NextGEMS/IFS_2.8km_deep_off/ifs_hpx2048_lsm_30-60N.nc')['lsm']
land_sea_ifs_cycle3 = xr.open_dataset('/home/jwille/xenon_data/NextGEMS/NextGEMS/IFS_4.4-FESOM_5/IFS_cycle3_lsm_30-60N.nc')['lsm']
land_sea_ifs_zoom9 = xr.open_dataset('/home/jwille/xenon_data/NextGEMS/NextGEMS/IFS_zoom9/30-60N_ifs_hpx512_lsm.nc')['lsm']
land_sea_icon_zoom10 = xr.open_dataset('/home/jwille/xenon_data/NextGEMS/NextGEMS/ICON_zoom10/icon_cycle3_lsm_30-60N.nc')['ocean_fraction_surface']
land_sea_icon_zoom9 = xr.open_dataset('/home/jwille/xenon_data/NextGEMS/NextGEMS/ICON_zoom9/30-60N_ICON_ngc4008_ocean_fraction_surface_zoom9.nc')['ocean_fraction_surface']
land_sea_imerg = xr.open_dataset('/home/jwille/xenon_data/NextGEMS/IMERG/clean/30-60N_fixgrid_GPM_IMERG_LandSeaMask.2.nc4')['landseamask']
land_sea_imerg = land_sea_imerg.assign_coords(lat=imerg.lat, lon=imerg.lon)

ifs_on_land = ifs_on.where(land_sea_ifs_high > 0, np.nan)
ifs_off_land = ifs_off.where(land_sea_ifs_high > 0, np.nan)
ifs_cycle3_land = ifs_cycle3.where(land_sea_ifs_cycle3 > 0, np.nan)
ifs_cycle4_land = ifs_cycle4.where(land_sea_ifs_zoom9 > 0, np.nan)
icon_cycle3_land = icon_cycle3.where(land_sea_icon_zoom10 < 1, np.nan)
icon_cycle4_land = icon_cycle4.where(land_sea_icon_zoom9 < 1, np.nan)
imerg_land = imerg.where(land_sea_imerg < 100, np.nan)


## CMIP6 multi-model distribution (replaces CESM2)

The ensemble files in `CMIP6/30-60N/ensemble` hold one file per model, with dimensions (member, year, lat, lon).
- **Per model:** only the first member is used (`CMIP6_MEMBERS_PER_MODEL = 1`; lowest r, then i, p, f, e.g. r1i1p1f1, or r2i1p1f1 if r1 does not exist), the same member as in `land_dry_spell_boxplot_CMIP6.ipynb`. All its years and land cells are pooled. Land is `sftlf > 0`, and each value gets a cos(lat) area weight.
- **Equal model weights:** each model's weights are scaled to sum to 1, so every model counts equally. The multi-model distribution is the average of the per-model distributions. For the KDE curve this equals the average of the per-model density curves.
- **Speed:** dry-spell lengths are whole days, so the weights are summed per distinct value, which gives identical quantiles. The KDE is given the bandwidth the full pooled sample would get (`cmip6_bw`), so the curve is the same as for the full sample.
- **Models without an sftlf file** (BCC-CSM2-MR): the mask of `FALLBACK_MASK_MODEL` is used, matched to the nearest grid point.

In [ ]:
import re
CMIP6_ENS_DIR       = '/home/jwille/xenon_data/NextGEMS/dry_spells/CMIP6/30-60N/ensemble'
SFTLF_DIR           = '/net/atmos/data/cmip6-ng/sftlf/fx/native'
FALLBACK_MASK_MODEL = 'MPI-ESM1-2-HR'
CMIP6_YEARS         = None      # e.g. (1985, 2014) to use only those years; None = all years (1850-2014)
CMIP6_EXCLUDE       = []        # model names to leave out, e.g. ['BCC-CSM2-MR']
CMIP6_MEMBERS_PER_MODEL = 1     # first N members of each model (sorted r, i, p, f); None = all members


def member_sort_key(member):
    return [int(n) for n in re.findall(r'\d+', member)]


def member_labels(ds):
    """Member names of an ensemble file: 'member_labels' attribute (NCO version, member = 1..N)
    or the member coordinate itself (Python version, member = 'r1i1p1f1', ...)."""
    lab = ds['member'].attrs.get('member_labels') or ds.attrs.get('member_labels')
    return lab.split(',') if lab else [str(m) for m in ds['member'].values]


def weighted_quantile(values, weights, quantiles):
    sorter = np.argsort(values)
    v = values[sorter]
    w = weights[sorter]
    cumw = np.cumsum(w)
    cumw /= cumw[-1]
    return np.interp(quantiles, cumw, v)


def weighted_quantile_hist(values, weights, quantiles):
    """Weighted quantile for aggregated data (distinct values + summed weights): the first value at which
    the cumulative weight reaches q. Equals weighted_quantile() applied to the full, non-aggregated sample
    (np.interp would otherwise interpolate between neighbouring whole-day values)."""
    sorter = np.argsort(values)
    v = np.asarray(values)[sorter]
    cumw = np.cumsum(np.asarray(weights)[sorter])
    cumw /= cumw[-1]
    return v[np.minimum(np.searchsorted(cumw, np.atleast_1d(quantiles) - 1e-12), v.size - 1)].squeeze()


def nearest_regrid(src, lat, lon):
    """Nearest-neighbour match of a lat/lon field to the given lat/lon (lon periodic).
    Identical grids give exactly the original values."""
    ilat = np.abs(src.lat.values[:, None] - np.asarray(lat)[None, :]).argmin(axis=0)
    dlon = (src.lon.values[:, None] - np.asarray(lon)[None, :] + 180) % 360 - 180
    ilon = np.abs(dlon).argmin(axis=0)
    return xr.DataArray(src.values[np.ix_(ilat, ilon)], dims=('lat', 'lon'), coords={'lat': lat, 'lon': lon})


def find_sftlf(model):
    """sftlf file of a model: historical preferred, else any experiment; None if there is none."""
    pat = re.compile(rf'^sftlf_fx_{re.escape(model)}_(?P<exp>[^_]+)_(?P<mem>[^_]+)_[^_]+\.nc$')
    cands = [(m['exp'] != 'historical', [int(n) for n in re.findall(r'\d+', m['mem'])], f)
             for f in os.listdir(SFTLF_DIR) if (m := pat.match(f))]
    return os.path.join(SFTLF_DIR, sorted(cands)[0][2]) if cands else None

In [ ]:
ens_pat = re.compile(r'^dry_spell_(?P<model>[^_]+)_historical_ensemble_[^_]+\.nc$')
ens_files = sorted((m['model'], f) for f in os.listdir(CMIP6_ENS_DIR) if (m := ens_pat.match(f)))
fallback_sftlf = xr.open_dataset(find_sftlf(FALLBACK_MASK_MODEL))['sftlf'].load()

cmip6_models = {}   # model -> dict(values, weights) : distinct values and their summed, normalised weights
sum_w2 = 0.0        # sum of squared per-sample weights (for the KDE bandwidth of the full pooled sample)
for model, fname in ens_files:
    if model in CMIP6_EXCLUDE:
        continue
    ds = xr.open_dataset(os.path.join(CMIP6_ENS_DIR, fname))
    labels = member_labels(ds)
    if len(labels) != ds.sizes['member']:
        raise ValueError(f'{model}: {len(labels)} member labels but {ds.sizes["member"]} members in {fname}')
    order = sorted(range(len(labels)), key=lambda i: member_sort_key(labels[i]))[:CMIP6_MEMBERS_PER_MODEL]
    members = [labels[i] for i in order]
    da = ds['dry_spell'].isel(member=order).load()                  # (member, year, lat, lon), first member(s) only
    if CMIP6_YEARS is not None:
        da = da.sel(year=slice(*CMIP6_YEARS))

    mask_path = find_sftlf(model)
    sftlf = nearest_regrid(xr.open_dataset(mask_path)['sftlf'].load() if mask_path else fallback_sftlf,
                           da.lat.values, da.lon.values)
    land = (sftlf > 0).values
    w2d = np.cos(np.deg2rad(da.lat.values))[:, None] * np.ones(da.sizes['lon'])

    vals = da.values[:, :, land]                                   # (member, year, n_land)
    wts = np.broadcast_to(w2d[land], vals.shape)
    ok = np.isfinite(vals)
    v, w = vals[ok], wts[ok]
    w = w / w.sum()                                                # every model sums to 1
    sum_w2 += np.sum(w ** 2)
    u, inv = np.unique(v, return_inverse=True)
    cmip6_models[model] = dict(values=u, weights=np.bincount(inv, weights=w), n_members=da.sizes['member'])
    print(f"{model:14s} {', '.join(members)}, {da.sizes['year']} years, {land.sum():5d} land cells"
          f"  mask: {os.path.basename(mask_path) if mask_path else FALLBACK_MASK_MODEL + ' (fallback)'}")

n_models = len(cmip6_models)
all_vals = np.unique(np.concatenate([d['values'] for d in cmip6_models.values()]))
mmm_w = np.zeros(all_vals.size)
for d in cmip6_models.values():
    mmm_w[np.searchsorted(all_vals, d['values'])] += d['weights'] / n_models

cmip6_mmm_land   = xr.DataArray(all_vals, dims='sample')           # distinct dry-spell lengths
w_cmip6_mmm_land = xr.DataArray(mmm_w,    dims='sample')           # multi-model weight of each length

# KDE bandwidth factor of the full pooled sample (Scott's rule with the effective sample size),
# corrected for the weighted-covariance estimate on the aggregated values
neff_full = 1.0 / (sum_w2 / n_models ** 2)
cmip6_bw = neff_full ** (-1 / 5) * np.sqrt((1 - np.sum(mmm_w ** 2)) / (1 - sum_w2 / n_models ** 2))
print(f'\n{n_models} models, CMIP6 multi-model KDE bandwidth factor {cmip6_bw:.4g}')

In [ ]:
imerg_land_mean = imerg_land.mean(dim=['lat','lon'])

In [ ]:
print(imerg_land_mean.std())

In [ ]:
print(imerg_land_mean.mean())

In [ ]:
ifs_cycle4_land_mean = ifs_cycle4_land.mean(dim=['value'])

In [ ]:
print(ifs_cycle4_land_mean.std())

In [ ]:
print(ifs_cycle4_land_mean.mean())

In [ ]:
cmip6_mmm_land_mean = np.average(cmip6_mmm_land, weights=w_cmip6_mmm_land)

In [ ]:
print(cmip6_mmm_land_mean)

In [ ]:
ifs_on_land = ifs_on_land.stack(sample=("year", "value"))
ifs_off_land = ifs_off_land.stack(sample=("year", "value"))
ifs_cycle3_land = ifs_cycle3_land.stack(sample=("year", "value"))
ifs_cycle4_land = ifs_cycle4_land.stack(sample=("year", "value"))
icon_cycle3_land = icon_cycle3_land.stack(sample=("year", "cell"))
icon_cycle4_land = icon_cycle4_land.stack(sample=("year", "cell"))
imerg_land = imerg_land.stack(sample=("year", "lat", "lon"))

In [ ]:
ifs_on_land = ifs_on_land.where(~ifs_on_land.isnull(), drop=True)
ifs_off_land = ifs_off_land.where(~ifs_off_land.isnull(), drop=True)
ifs_cycle3_land = ifs_cycle3_land.where(~ifs_cycle3_land.isnull(), drop=True)
ifs_cycle4_land = ifs_cycle4_land.where(~ifs_cycle4_land.isnull(), drop=True)
icon_cycle3_land = icon_cycle3_land.where(~icon_cycle3_land.isnull(), drop=True)
icon_cycle4_land = icon_cycle4_land.where(~icon_cycle4_land.isnull(), drop=True)
imerg_land = imerg_land.where(~imerg_land.isnull(), drop=True)


In [ ]:
#For IMERG weighting
# 1D cos(lat) weights
lat_rad = np.deg2rad(imerg.lat)
coslat = np.cos(lat_rad)                   # dims: ['lat']

w_lat = xr.DataArray(
    coslat,
    dims=['lat'],
    coords={'lat': imerg.lat}
)

# Broadcast to (lat, lon)
w_imerg_2d = w_lat.broadcast_like(land_sea_imerg)
# Mask to land only (same land mask you use for imerg_land)
w_imerg_2d = w_imerg_2d.where(land_sea_imerg < 100)
#w_imerg_2d = w_imerg_2d / w_imerg_2d.sum()
# Broadcast spatial weights to (year, lat, lon):
w_imerg_full = xr.DataArray(
    np.broadcast_to(w_imerg_2d.values, imerg.shape),
    dims=imerg.dims,
    coords=imerg.coords,
)
w_imerg_land = w_imerg_full.where(land_sea_imerg < 100, np.nan)

# Stack in exactly the same way as imerg_land
w_imerg_land = w_imerg_land.stack(sample=("year", "lat", "lon"))

# Drop weights where the data is NaN to keep alignment
w_imerg_land = w_imerg_land.where(~imerg_land.isnull(), drop=True)
x_imerg = imerg_land.values.ravel()
w_imerg = w_imerg_land.values.ravel()


In [ ]:
print(w_imerg_2d.sum())

In [ ]:
# CMIP6 multi-model weights (w_cmip6_mmm_land) are computed in the CMIP6 cells above

In [ ]:
print(np.median(imerg_land))


In [ ]:
print(weighted_quantile_hist(cmip6_mmm_land.values, w_cmip6_mmm_land.values, 0.5))

In [ ]:
print(np.median(ifs_off_land))

In [ ]:
print(np.median(ifs_on_land))

In [ ]:
print(np.median(icon_cycle4_land))

In [ ]:
print(np.median(icon_cycle3_land))

In [ ]:
print(np.median(ifs_cycle3_land))

In [ ]:
print(np.median(ifs_cycle4_land))

In [ ]:
print(np.quantile(ifs_on_land,0.95))

In [ ]:
print(np.quantile(ifs_off_land,0.95))

In [ ]:
print(np.quantile(ifs_cycle3_land,0.95))

In [ ]:
print(np.quantile(ifs_cycle4_land,0.95))

In [ ]:
print(np.quantile(icon_cycle3_land,0.95))

In [ ]:
print(np.quantile(icon_cycle4_land,0.95))

In [ ]:
print(weighted_quantile_hist(cmip6_mmm_land.values, w_cmip6_mmm_land.values, 0.95))

In [ ]:
print(np.quantile(imerg_land,0.95))

In [ ]:
print(imerg_land)

In [ ]:
print(ifs_on_land)

In [ ]:
print(icon_cycle4_land.max())

In [ ]:
print(cmip6_mmm_land.max())

In [ ]:
mask = (ifs_on_land == 184) 
percentage = (mask.sum() / ifs_on_land.size) * 100
print(f"Percentage equal to 184: {percentage.values:.2f}%")

In [ ]:
mask = (ifs_off_land == 184)
percentage = (mask.sum() / ifs_off_land.size) * 100
print(f"Percentage  equal to 184: {percentage.values:.2f}%")

In [ ]:
mask = (ifs_cycle3_land == 184)
percentage = (mask.sum() / ifs_cycle3_land.size) * 100
print(f"Percentage  equal to 184: {percentage.values:.2f}%")

In [ ]:
mask = (ifs_cycle4_land == 184)
percentage = (mask.sum() / ifs_cycle4_land.size) * 100
print(f"Percentage  equal to 184: {percentage.values:.2f}%")

In [ ]:
mask = (icon_cycle3_land == 184)
percentage = (mask.sum() / icon_cycle3_land.size) * 100
print(f"Percentage  equal to 184: {percentage.values:.2f}%")

In [ ]:
mask = (icon_cycle4_land == 184)
percentage = (mask.sum() / icon_cycle4_land.size) * 100
print(f"Percentage  equal to 184: {percentage.values:.2f}%")

In [ ]:
mask = (cmip6_mmm_land == 184)
percentage = (w_cmip6_mmm_land.where(mask, 0).sum() / w_cmip6_mmm_land.sum()) * 100
print(f"Percentage  equal to 184: {percentage.values:.2f}%")

In [ ]:
mask = (imerg == 184)
percentage = (mask.sum() / imerg.size) * 100
print(f"Percentage equal to 184: {percentage.values:.2f}%")

In [ ]:
mask = (ifs_on_land > 60) & (ifs_on_land < 185)
percentage = (mask.sum() / ifs_on_land.size) * 100
print(f"Percentage between 60 and 184: {percentage.values:.2f}%")



In [ ]:
mask = (ifs_off_land > 60) & (ifs_off_land < 185)
percentage = (mask.sum() / ifs_off_land.size) * 100
print(f"Percentage between 60 and 184: {percentage.values:.2f}%")



In [ ]:
mask = (ifs_cycle3_land > 60) & (ifs_cycle3_land < 185)
percentage = (mask.sum() / ifs_cycle3_land.size) * 100
print(f"Percentage between 60 and 184: {percentage.values:.2f}%")



In [ ]:
mask = (ifs_cycle4_land > 60) & (ifs_cycle4_land < 185)
percentage = (mask.sum() / ifs_cycle4_land.size) * 100
print(f"Percentage between 60 and 184: {percentage.values:.2f}%")



In [ ]:
mask = (icon_cycle3_land > 60) & (icon_cycle3_land < 185)
percentage = (mask.sum() / icon_cycle3_land.size) * 100
print(f"Percentage between 60 and 184: {percentage.values:.2f}%")



In [ ]:
mask = (icon_cycle4_land > 60) & (icon_cycle4_land < 185)
percentage = (mask.sum() / icon_cycle4_land.size) * 100
print(f"Percentage between 60 and 184: {percentage.values:.2f}%")



In [ ]:
mask = (cmip6_mmm_land > 60) & (cmip6_mmm_land < 185)
percentage = (w_cmip6_mmm_land.where(mask, 0).sum() / w_cmip6_mmm_land.sum()) * 100
print(f"Percentage between 60 and 184: {percentage.values:.2f}%")


In [ ]:
mask = (imerg_land > 60) & (imerg_land < 185)
percentage = (mask.sum() / imerg_land.size) * 100
print(f"Percentage between 60 and 184: {percentage.values:.2f}%")



In [ ]:
# For equal-area grids: weights = 1
def ones_like_da(da):
    return xr.ones_like(da)

In [ ]:
#with weights

# 1) Common x-range + bin width
xmin, xmax = 0, 185


bin_width  = 2.0






datasets = [
    # IFS/ICON: equal-area → uniform weights
    (ifs_on_land,       ones_like_da(ifs_on_land),       '#FCC5E4', 'IFS_2.8_deep_on'),
    (ifs_off_land,      ones_like_da(ifs_off_land),      '#F768A1', 'IFS_2.8_deep_off'),
    (ifs_cycle3_land,   ones_like_da(ifs_cycle3_land),   '#C51B8A', 'IFS_4.4-FESOM_5'),
    (ifs_cycle4_land,   ones_like_da(ifs_cycle4_land),   '#7A0177', 'IFS_9-FESOM_5'),
    (icon_cycle3_land,  ones_like_da(icon_cycle3_land),  '#4DBBEE', 'ICON_5'),
    (icon_cycle4_land,  ones_like_da(icon_cycle4_land),  '#0A5FA3', 'ICON_9'),

    # CMIP6 multi-model & IMERG: area weights from above
    (cmip6_mmm_land,    w_cmip6_mmm_land,                '#D62728', 'CMIP6'),
    (imerg_land,        w_imerg_land,                    '#000000',       'IMERG'),
]



plt.figure(figsize=(16, 8))
sns.set_style("whitegrid")
plt.rc('axes', titlesize=22)
plt.rc('axes', labelsize=18)

for data_da, wts_da, color, label in datasets:
    x = data_da.values.ravel()
    w = wts_da.values.ravel()

    sns.kdeplot(
        x=x,
        weights=w,
        bw_method=cmip6_bw if label == 'CMIP6' else 'scott',   # CMIP6: bandwidth of the full pooled sample
        bw_adjust=1.0,
        clip=(xmin, xmax),
        common_norm=False,
        fill=False,
        linewidth=2.0,
        color=color,
        label=label
    )

# 4) Rescale each line from density → percent per 0.5% bin
ax = plt.gca()
for line in ax.get_lines():
    x_vals, y_vals = line.get_data()
    y_scaled = y_vals * 100 * bin_width
    line.set_data(x_vals, y_scaled)

# 5) Labels, limits, legend
xl = 'Longest dry spell length (days)'
ax.set_ylim(0, 10)
plt.xlabel(xl, fontsize=24)

plt.ylabel('Percentage of total area (%)', fontsize=24)


plt.xlim(xmin, xmax)
plt.tick_params(axis='both', which='major', labelsize=22)

xticks = np.arange(xmin, xmax, 10)
ax.set_xticks(xticks)
# build a list of Line2D handles matching your plotted lines
legend_handles = [
    Line2D([0], [0], color=color, linewidth=2.0)
    for *_, color, _ in datasets
]
# extract the labels out of your datasets tuple
legend_labels = [label for *_, _, label in datasets]

# then call legend with those handles
plt.legend(
    legend_handles,
    legend_labels,
    loc='upper right',
    prop={'size': 22},
    bbox_to_anchor=(1.0, 1.0),
    handlelength=2.5  # longer lines in legend if you like
)

plt.tight_layout()
plt.savefig('/home/jwille/images/global_paper/mid_lat_land_dry_spell_histo_CMIP6mmm.pdf', format='pdf',bbox_inches='tight')

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

# ── weighted quantile helper ──────────────────────────────────────────────────
def weighted_quantile(values, weights, quantiles):
    sorter = np.argsort(values)
    v = values[sorter]
    w = weights[sorter]
    cumw = np.cumsum(w)
    cumw /= cumw[-1]
    return np.interp(quantiles, cumw, v)

# ── datasets: IFS/ICON use uniform weights, CMIP6/IMERG use cos(lat) ─────────
datasets = [
    (ifs_on_land,      ones_like_da(ifs_on_land),      '#FCC5E4', 'IFS_2.8_deep_on'),
    (ifs_off_land,     ones_like_da(ifs_off_land),     '#F768A1', 'IFS_2.8_deep_off'),
    (ifs_cycle3_land,  ones_like_da(ifs_cycle3_land),  '#C51B8A', 'IFS_4.4-FESOM_5'),
    (ifs_cycle4_land,  ones_like_da(ifs_cycle4_land),  '#7A0177', 'IFS_9-FESOM_5'),
    (icon_cycle3_land, ones_like_da(icon_cycle3_land), '#4DBBEE', 'ICON_5'),
    (icon_cycle4_land, ones_like_da(icon_cycle4_land), '#0A5FA3', 'ICON_9'),
    (cmip6_mmm_land,   w_cmip6_mmm_land,               '#D62728', 'CMIP6'),
    (imerg_land,       w_imerg_land,                   '#000000', 'IMERG'),
]

# ── collect stats ─────────────────────────────────────────────────────────────
labels, colors, stats = [], [], []

for data_da, wts_da, color, label in datasets:
    x = data_da.values.ravel().astype(float)
    w = wts_da.values.ravel().astype(float)

    # drop NaNs (uniform-weight arrays have no NaNs, but guard anyway)
    mask = np.isfinite(x) & np.isfinite(w)
    x, w = x[mask], w[mask]

    wq = weighted_quantile_hist if label == 'CMIP6' else weighted_quantile   # CMIP6 is aggregated
    q25, q50, q75 = wq(x, w, [0.25, 0.50, 0.75])
    iqr       = q75 - q25
    x_w       = x[w > 0]                                   # values that occur
    lo_whisk  = x_w[x_w >= q25 - 1.5 * iqr].min()
    hi_whisk  = x_w[x_w <= q75 + 1.5 * iqr].max()
    mean      = np.average(x, weights=w)

    labels.append(label)
    colors.append(color)
    stats.append(dict(q25=q25, q50=q50, q75=q75,
                      lo=lo_whisk, hi=hi_whisk, mean=mean))

# ── plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 7))
sns.set_style("whitegrid")

n   = len(stats)
pos = np.arange(1, n + 1)
bw  = 0.45

for p, s, c in zip(pos, stats, colors):
    # IQR box
    rect = mpatches.FancyBboxPatch(
        (p - bw, s['q25']), 2 * bw, s['q75'] - s['q25'],
        boxstyle="square,pad=0", linewidth=1.5,
        edgecolor='k', facecolor=c, alpha=0.85, zorder=3
    )
    ax.add_patch(rect)

    # median line
    ax.hlines(s['q50'], p - bw, p + bw, colors='k', linewidth=2.0, zorder=4)

    # whiskers
    ax.vlines(p, s['lo'],  s['q25'], colors='k', linewidth=1.5, zorder=2)
    ax.vlines(p, s['q75'], s['hi'],  colors='k', linewidth=1.5, zorder=2)

    # whisker caps
    cap_hw = bw * 0.45
    ax.hlines(s['lo'], p - cap_hw, p + cap_hw, colors='k', linewidth=1.5, zorder=2)
    ax.hlines(s['hi'], p - cap_hw, p + cap_hw, colors='k', linewidth=1.5, zorder=2)

    # weighted mean diamond
    #ax.plot(p, s['mean'], marker='D', color='white',
      #      markeredgecolor='k', markersize=7, zorder=5)

# ── axes formatting ───────────────────────────────────────────────────────────
ax.set_xlim(0.3, n + 0.7)
ax.set_ylim(0, 185)
ax.set_xticks(pos)
ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=18)
ax.set_ylabel('Longest dry spell length (days)', fontsize=20)
ax.set_yticks(np.arange(0, 190, 20))
ax.tick_params(axis='y', labelsize=16)

# ── legend ────────────────────────────────────────────────────────────────────
legend_elements = [
    Line2D([0], [0], color='k', linewidth=2,                           label='Median'),
    Line2D([0], [0], marker='D', color='w', markerfacecolor='white',
           markeredgecolor='k', markersize=8,                          label='Weighted mean'),
    mpatches.Patch(facecolor='grey', edgecolor='k', alpha=0.85,        label='IQR (25–75 %)'),
    Line2D([0], [0], color='k', linewidth=1.5,                         label='Whiskers (1.5 × IQR)'),
]
#ax.legend(handles=legend_elements, loc='upper right', fontsize=13, framealpha=0.9)

plt.tight_layout()
plt.savefig('/home/jwille/images/global_paper/mid_lat_land_dry_spell_boxplot_CMIP6mmm.pdf',
            format='pdf', bbox_inches='tight')
plt.show()